In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers_bronze = spark.table(
    "retail_bronze.customers"
)

print("Bronze customer count:", customers_bronze.count())

display(customers_bronze.limit(10))

In [0]:
duplicate_ids = (
    customers_bronze
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:", duplicate_ids.count())

display(duplicate_ids.limit(20))

In [0]:
# Separate invalid records before deduplication

invalid_customers = (
    customers_bronze
    .filter(F.col("customer_id").isNull())
)

valid_customer_candidates = (
    customers_bronze
    .filter(F.col("customer_id").isNotNull())
)

print("Invalid customer records:", invalid_customers.count())
print("Valid candidates:", valid_customer_candidates.count())

In [0]:
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("updated_at").desc())
)

customers_ranked = (
    valid_customer_candidates
    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )
)

display(
    customers_ranked
    .filter(F.col("customer_id") == 1)
)

In [0]:
customers_deduped = (
    customers_ranked
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print(
    "After deduplication:",
    customers_deduped.count()
)

display(customers_deduped.limit(10))

In [0]:
valid_customers = customers_deduped

print(
    "Valid customer records:",
    valid_customers.count()
)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS retail_silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS retail_quarantine
""")

In [0]:
valid_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.customers")

In [0]:
invalid_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_quarantine.customers")

In [0]:
print(
    "Bronze:",
    customers_bronze.count()
)

print(
    "Quarantine:",
    spark.table("retail_quarantine.customers").count()
)

print(
    "Silver:",
    spark.table("retail_silver.customers").count()
)